In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs
metric = pd.read_csv('metric.csv')
trace = pd.read_csv('trace.csv')
log = pd.read_csv('log.csv')
error_logs = pd.read_csv('error_logs.csv')

# Ensure numeric types where applicable
for df, valcol in [(metric, 'value'), (trace, 'value'), (log, 'value')]:
    if valcol in df.columns:
        df[valcol] = pd.to_numeric(df[valcol], errors='coerce')

# Add localized datetime column (UTC+8) for inspection (do not change original timestamp semantics)
for df in [metric, trace, log, error_logs]:
    if 'timestamp' in df.columns:
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) General counts
counts_df = pd.DataFrame({
    'file': ['metric.csv', 'trace.csv', 'log.csv', 'error_logs.csv'],
    'rows': [metric.shape[0], trace.shape[0], log.shape[0], error_logs.shape[0]]
})

# 2) Components summary: sorted unique cmdb_id list (limit display to first 50) and counts
def comp_summary(df):
    uniq = sorted(df['cmdb_id'].dropna().unique())
    sample = uniq[:50]
    return pd.Series({'unique_count': len(uniq), 'cmdb_ids_sample': sample})

components_summary = pd.DataFrame({
    'metric': comp_summary(metric),
    'trace': comp_summary(trace),
    'log': comp_summary(log),
    'error_logs': comp_summary(error_logs)
}).T.reset_index().rename(columns={'index':'file'})

# 3) Metric analysis
# grouped object stored as metric_agg (keeps raw points accessible via the groupby object)
metric_agg = metric.groupby(['cmdb_id', 'kpi_name'])

# number of unique pairs
metric_unique_pairs = metric_agg.ngroups

# aggregated stats for all groups (rows and global p95)
metric_stats_all = metric_agg['value'].agg(rows='size', p95_value=lambda s: s.quantile(0.95)).reset_index()

# top 20 by rows
metric_top20 = metric_stats_all.sort_values('rows', ascending=False).head(20).reset_index(drop=True)

# 4) Trace analysis
trace_agg = trace.groupby(['cmdb_id', 'trace_name'])
trace_unique_pairs = trace_agg.ngroups
trace_stats_all = trace_agg['value'].agg(rows='size', p95_value=lambda s: s.quantile(0.95)).reset_index()
trace_top20 = trace_stats_all.sort_values('rows', ascending=False).head(20).reset_index(drop=True)

# 5) Log analysis
log_agg = log.groupby(['cmdb_id', 'log_name'])
log_unique_pairs = log_agg.ngroups
log_stats_all = log_agg['value'].agg(rows='size', p95_value=lambda s: s.quantile(0.95)).reset_index()
log_top20 = log_stats_all.sort_values('rows', ascending=False).head(20).reset_index(drop=True)

# 6) Error logs: up to first 20 rows for inspection (select requested columns)
error_head = error_logs[['timestamp', 'cmdb_id', 'message', 'dt']].head(20).copy()

# Compact outputs (limited to requested columns / rows)
counts_df, components_summary, metric_unique_pairs, metric_top20[['cmdb_id','kpi_name','rows','p95_value']], trace_unique_pairs, trace_top20[['cmdb_id','trace_name','rows','p95_value']], log_unique_pairs, log_top20[['cmdb_id','log_name','rows','p95_value']], error_head

```
Out[1]:
```
```python
summary = (
    "Telemetry load summary:\n"
    "- Row counts: metric.csv=28015, trace.csv=6834, log.csv=511, error_logs.csv=52.\n"
    "- Unique components: metric=29, trace=12, log=7, error_logs=3.\n\n"
    "Metric highlights:\n"
    "- Unique (cmdb_id,kpi_name) pairs: 1694.\n"
    "- Top metric pairs (examples from top 20):\n"
    "  * apache01 - OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies: 31 rows, p95=1.5\n"
    "  * IG01 - JVM-Threads_7778_JVM_ThreadCount_Threads: 31 rows, p95≈74.0\n"
    "  * IG01 - JVM-Memory_7778_JVM_Memory_HeapMemoryUsed: 31 rows, p95≈8.24e8\n\n"
    "Trace highlights:\n"
    "- Unique (cmdb_id,trace_name) pairs: 261.\n"
    "- Top trace pairs (examples from top 20):\n"
    "  * IG02 - trace.self.row_count: 31 rows, p95≈1772.5\n"
    "  * IG02 - trace.self.duration_mean: 31 rows, p95≈0.693\n"
    "  * IG01 - trace.self.duration_mean: 30 rows, p95≈0.708\n\n"
    "Log highlights:\n"
    "- Unique (cmdb_id,log_name) pairs: 18.\n"
    "- Top log pairs (examples from top 20):\n"
    "  * Tomcat01 - log.row_count: 31 rows, p95≈442\n"
    "  * Tomcat01 - log.error_count: 31 rows, p95=0\n"
    "  * apache02 - log.row_count: 31 rows, p95≈1030\n\n"
    "Error logs:\n"
    "- First 20 error log rows were inspected: entries come from Tomcat04, apache01, apache02 with timestamps around 2021-03-07 16:13–16:29 (Asia/Shanghai, UTC+8).\n\n"
    "Notes:\n"
    "- Grouped DataFrames with raw points are saved as kernel variables: metric_agg, trace_agg, log_agg for follow-up analysis."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(             file   rows
0      metric.csv  28015
1       trace.csv   6834
2         log.csv    511
3  error_logs.csv     52,          file unique_count                                    cmdb_ids_sample
0      metric           29  [IG01, IG02, MG01, MG02, Mysql01, Mysql02, Red...
1       trace           12  [IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, T...
2         log            7  [Mysql01, Tomcat01, Tomcat02, Tomcat03, Tomcat...
3  error_logs            3                     [Tomcat04, apache01, apache02], 1694,      cmdb_id                                           kpi_name  rows     p95_value
0   apache01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    31  1.500000e+00
1       IG01           JVM-Threads_7778_JVM_ThreadCount_Threads    31  7.400000e+01
2   apache02                 OSLinux-OSLinux_ZABBIX_Host_Uptime    31  6.304393e+06
3       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed    31  5.817198e+07
4   apache01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    31  4.000000e+00
5       IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed    31  8.241607e+08
6   apache02                            OSLinux-CPU_CPU_CPULoad    31  8.500000e-02
7   apache02                         OSLinux-CPU_CPU_CPUSysTime    31  4.424860e-01
8   apache02                        OSLinux-CPU_CPU_CPUUserTime    31  2.223000e-01
9   apache01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...    31  5.000000e-01
10  apache02                        OSLinux-CPU_CPU_CPUidleutil    31  9.954925e+01
11   Mysql02                        OSLinux-CPU_CPU_CPUUserTime    31  2.787050e+00
12   Mysql02                             OSLinux-CPU_CPU_CPUWio    31  1.334120e+01
13      IG01                            OSLinux-CPU_CPU_CPULoad    31  1.135000e+00
14      IG01                         OSLinux-CPU_CPU_CPUSysTime    31  4.907030e-01
15      IG01                        OSLinux-CPU_CPU_CPUUserTime    31  2.535451e+01
16      IG01                             OSLinux-CPU_CPU_CPUWio    31  2.100000e-01
17      IG01                        OSLinux-CPU_CPU_CPUidleutil    31  7.417760e+01
18  apache02                             OSLinux-CPU_CPU_CPUWio    31  1.495500e-01
19   Redis01             redis-Redis_6379_Redis  (expired_keys)    31  6.000000e+00, 261,      cmdb_id                       trace_name  rows    p95_value
0       IG02             trace.self.row_count    31  1772.500000
1       IG02         trace.self.duration_mean    31     0.692731
2       IG02          trace.self.duration_p95    31     3.126875
3   dockerB2          trace.to_MG02.row_count    30   301.850000
4       IG01  trace.to_Tomcat01.duration_mean    30     0.720486
5       IG01         trace.self.duration_mean    30     0.708116
6       IG01          trace.self.duration_p95    30     3.174392
7       IG01             trace.self.row_count    30  1778.800000
8   dockerB1    trace.from_MG02.duration_mean    30     0.954452
9   dockerA2         trace.self.duration_mean    30     0.079310
10  dockerA2          trace.self.duration_p95    30     1.014000
11  dockerA2             trace.self.row_count    30  2766.350000
12  dockerA1       trace.to_MG02.duration_p95    30     1.035390
13  dockerA2      trace.to_MG02.duration_mean    30     0.113542
14  dockerA2       trace.to_MG02.duration_p95    30     1.012705
15  dockerA2          trace.to_MG02.row_count    30    63.550000
16  dockerA2        trace.from_MG01.row_count    30   196.050000
17  dockerA2    trace.from_MG02.duration_mean    30     0.571160
18  dockerA2     trace.from_MG02.duration_p95    30     2.087222
19  dockerA2        trace.from_MG02.row_count    30   276.900000, 18,      cmdb_id          log_name  rows  p95_value
0   Tomcat01     log.row_count    31     442.00
1   Tomcat01   log.error_count    31       0.00
2   apache02   log.error_count    31       0.00
3   apache01     log.row_count    31     740.50
4   Tomcat02   log.error_count    31       0.00
5   Tomcat02     log.row_count    31     434.50
6   Tomcat03   log.error_count    31       0.00
7   Tomcat04     log.row_count    31     463.00
8   Tomcat04   log.error_count    31       0.00
9   Tomcat03     log.row_count    31     453.50
10  apache02     log.row_count    31    1030.00
11  apache01   log.error_count    31       0.00
12  Tomcat02  log.gc_pause_max    30       0.10
13   Mysql01   log.error_count    23       0.00
14   Mysql01     log.row_count    23       4.00
15  Tomcat01  log.gc_pause_max    21       0.10
16  Tomcat03  log.gc_pause_max    21       0.09
17  Tomcat04  log.gc_pause_max    21       0.08,      timestamp   cmdb_id                                            message                        dt
0   1615104822  Tomcat04  IG02 POST /UOCP/person/ServiceTest9.json HTTP/... 2021-03-07 16:13:42+08:00
1   1615105743  apache01  IPAddress "POST /UOCP/person/ServiceTest10.jso... 2021-03-07 16:29:03+08:00
2   1615105743  apache02  IPAddress "POST /UOCP/person/ServiceTest1.json... 2021-03-07 16:29:03+08:00
3   1615105745  apache02  IPAddress "POST /UOCP/person/ServiceTest5.json... 2021-03-07 16:29:05+08:00
4   1615105746  apache01  IPAddress "POST /UOCP/base/ServiceTest6.json H... 2021-03-07 16:29:06+08:00
5   1615105746  apache01  IPAddress "POST /UOCP/person/ServiceTest3.json... 2021-03-07 16:29:06+08:00
6   1615105747  apache01  IPAddress "POST /UOCP/person/ServiceTest9.json... 2021-03-07 16:29:07+08:00
7   1615105747  apache02  IPAddress "POST /UOCP/base/ServiceTest11.json ... 2021-03-07 16:29:07+08:00
8   1615105747  apache02  IPAddress "POST /UOCP/base/ServiceTest8.json H... 2021-03-07 16:29:07+08:00
9   1615105749  apache01  IPAddress "POST /UOCP/base/ServiceTest8.json H... 2021-03-07 16:29:09+08:00
10  1615105749  apache02  IPAddress "POST /UOCP/base/ServiceTest7.json H... 2021-03-07 16:29:09+08:00
11  1615105749  apache02  IPAddress "POST /UOCP/person/ServiceTest5.json... 2021-03-07 16:29:09+08:00
12  1615105750  apache01  IPAddress "POST /UOCP/base/ServiceTest4.json H... 2021-03-07 16:29:10+08:00
13  1615105751  apache01  IPAddress "POST /UOCP/base/ServiceTest6.json H... 2021-03-07 16:29:11+08:00
14  1615105751  apache02  IPAddress "POST /UOCP/base/ServiceTest11.json ... 2021-03-07 16:29:11+08:00
15  1615105751  apache02  IPAddress "POST /UOCP/person/ServiceTest5.json... 2021-03-07 16:29:11+08:00
16  1615105751  apache02  IPAddress "POST /UOCP/base/ServiceTest2.json H... 2021-03-07 16:29:11+08:00
17  1615105753  apache02  IPAddress "POST /UOCP/base/ServiceTest6.json H... 2021-03-07 16:29:13+08:00
18  1615105753  apache01  IPAddress "POST /UOCP/person/ServiceTest5.json... 2021-03-07 16:29:13+08:00
19  1615105754  apache02  IPAddress "POST /UOCP/person/ServiceTest9.json... 2021-03-07 16:29:14+08:00)```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing 'metric' DataFrame assumed in kernel.
# 1) Compute global P95 and P05 per (cmdb_id, kpi_name)
metric_thresholds = (
    metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(p95=lambda s: s.quantile(0.95), p05=lambda s: s.quantile(0.05), rows='size')
    .reset_index()
)

# 2) Filter metric rows to incident time window and store
metric_incident_filtered = metric[
    (metric['timestamp'] >= 1615104000) & (metric['timestamp'] <= 1615105800)
].copy()
# add localized datetime for convenience (keep timestamp semantics)
metric_incident_filtered['dt'] = pd.to_datetime(metric_incident_filtered['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 3) Identify anomaly points by joining thresholds and applying conditions
m = metric_incident_filtered.merge(metric_thresholds, on=['cmdb_id', 'kpi_name'], how='left', suffixes=('','_thr'))

# define anomaly_type: HIGH if value >= p95, else LOW if value <= p05, else None
m['anomaly_type'] = np.where(m['value'] >= m['p95'], 'HIGH',
                             np.where(m['value'] <= m['p05'], 'LOW', None))

anomalies = m[m['anomaly_type'].notna()].copy()

# compute severity safely (avoid division by zero)
eps = 1e-12
def compute_severity(row):
    if row['anomaly_type'] == 'HIGH':
        denom = row['p95'] if pd.notnull(row['p95']) else np.nan
        if denom == 0 or pd.isna(denom):
            return np.nan
        return row['value'] / denom
    else:  # LOW
        denom = row['value'] if pd.notnull(row['value']) else np.nan
        if denom == 0 or pd.isna(denom):
            return np.nan
        return row['p05'] / denom

anomalies['severity_ratio'] = anomalies.apply(compute_severity, axis=1)

# 4) Aggregate anomalies by (cmdb_id, kpi_name, anomaly_type)
# rows_in_window = total rows for that (cmdb_id,kpi_name) within filtered window
rows_in_window_series = metric_incident_filtered.groupby(['cmdb_id','kpi_name']).size()
rows_in_window_df = rows_in_window_series.reset_index().rename(columns={0:'rows_in_window'})

agg = (
    anomalies
    .groupby(['cmdb_id','kpi_name','anomaly_type'])
    .agg(
        anomaly_count=('severity_ratio','size'),
        earliest_anomaly_timestamp=('timestamp','min'),
        max_severity_ratio=('severity_ratio','max')
    )
    .reset_index()
)

# attach rows_in_window
agg = agg.merge(rows_in_window_df, on=['cmdb_id','kpi_name'], how='left')

# keep requested columns and sort by severity desc, limit to top20
agg_sorted = agg.sort_values('max_severity_ratio', ascending=False).reset_index(drop=True)
anomalies_agg_top20 = agg_sorted[['cmdb_id','kpi_name','anomaly_type','anomaly_count','earliest_anomaly_timestamp','max_severity_ratio','rows_in_window']].head(20)

# For compact display of stored kernel variables, prepare small previews
metric_thresholds_head = metric_thresholds.head(20)
metric_incident_filtered_shape = metric_incident_filtered.shape

# Return: aggregated top20 (compact), and keep metric_thresholds & metric_incident_filtered stored for follow-up.
anomalies_agg_top20, metric_thresholds_head, metric_incident_filtered_shape

```
Out[2]:
```
```python
# Summarize the previous execution results in plain English and return as a single string.
try:
    total_thresholds = metric_thresholds.shape[0]
except NameError:
    total_thresholds = 'unknown'

try:
    rows_in_window = metric_incident_filtered.shape[0]
except NameError:
    rows_in_window = 'unknown'

try:
    total_anomaly_groups = anomalies.groupby(['cmdb_id','kpi_name','anomaly_type']).ngroups
except NameError:
    # fallback: use top20 length if anomalies/agg not available
    total_anomaly_groups = anomalies_agg_top20.shape[0] if 'anomalies_agg_top20' in globals() else 'unknown'

# Prepare top 5 anomaly group summaries if available
top_lines = []
if 'anomalies_agg_top20' in globals() and len(anomalies_agg_top20) > 0:
    top5 = anomalies_agg_top20.head(5)
    for i, row in top5.reset_index(drop=True).iterrows():
        top_lines.append(
            f"{i+1}. {row['cmdb_id']} - {row['kpi_name']} [{row['anomaly_type']}], "
            f"count={int(row['anomaly_count'])}, earliest_ts={int(row['earliest_anomaly_timestamp'])}, "
            f"max_severity={float(row['max_severity_ratio']):.3f}, rows_in_window={int(row['rows_in_window'])}"
        )
else:
    top_lines.append("No anomaly groups found in the filtered window.")

summary = (
    f"Summary of metric anomaly detection:\n"
    f"- Thresholds computed for {total_thresholds} unique (cmdb_id,kpi_name) pairs and stored in 'metric_thresholds'.\n"
    f"- Metric rows in the incident window (timestamps 1615104000..1615105800): {rows_in_window} rows stored in 'metric_incident_filtered'.\n"
    f"- Detected anomaly groups in window: {total_anomaly_groups} groups (aggregated by cmdb_id, kpi_name, anomaly_type).\n"
    f"- Top anomaly groups by severity (top 5):\n  " + "\n  ".join(top_lines) + "\n\n"
    f"Notes:\n"
    f"- HIGH anomalies were flagged when value >= global P95; LOW when value <= global P05 (thresholds computed from full historical series).\n"
    f"- The grouped thresholds and the filtered incident data are available in kernel variables 'metric_thresholds' and 'metric_incident_filtered' for follow-up.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name anomaly_type  anomaly_count  earliest_anomaly_timestamp  max_severity_ratio  rows_in_window
0       IG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT         HIGH              2                  1615105260          124.000000              31
1   Tomcat04       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT         HIGH              2                  1615104720           51.000000              21
2   Tomcat01                    OSLinux-CPU_CPU-2_SingleCpuidle          LOW              1                  1615105620           37.805295               9
3       MG01    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec          LOW              1                  1615104720           26.220690               9
4       MG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT         HIGH              1                  1615104240           20.000000              20
5       MG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT         HIGH              1                  1615104780           20.000000              20
6       MG01        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn          LOW              1                  1615104720           13.076923               9
7       MG01       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut          LOW              1                  1615104720           12.696000               9
8   Tomcat04                    OSLinux-CPU_CPU-1_SingleCpuUtil          LOW              1                  1615104420           10.236331              10
9       MG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT         HIGH              1                  1615104720           10.000000              20
10   Redis01                    OSLinux-CPU_CPU-2_SingleCpuUtil          LOW              1                  1615105500            9.919845              13
11  Tomcat04                    OSLinux-CPU_CPU-3_SingleCpuidle          LOW              1                  1615104300            9.529339               9
12      IG02                             OSLinux-CPU_CPU_CPUWio         HIGH              2                  1615104120            7.043732              31
13   Mysql02               Mysql-MySQL_3306_Max trx rows locked         HIGH              3                  1615104360            7.000000              21
14  apache01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT          LOW              4                  1615104060            6.000000              29
15  Tomcat03                    OSLinux-CPU_CPU-0_SingleCpuidle          LOW              1                  1615105140            5.608666               9
16  Tomcat01                    OSLinux-CPU_CPU-0_SingleCpuUtil          LOW              1                  1615105620            5.253245              10
17  apache01                    OSLinux-CPU_CPU-2_SingleCpuidle          LOW              1                  1615104900            5.150028              15
18   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead         HIGH              1                  1615104000            4.000000              16
19   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...         HIGH              1                  1615105320            4.000000              16,    cmdb_id                                         kpi_name           p95           p05  rows
0     IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09    31
1     IG01       JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  4.018995e+01  2.491370e+00    20
2     IG01        JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.241607e+08  8.685826e+07    31
3     IG01      JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.817198e+07  5.815991e+07    31
4     IG01        JVM-Operating System_7778_JVM_JVM_CPULoad  4.537150e-01  4.000600e-01    20
5     IG01                  JVM-Runtime_7778_JVM_JVM_Uptime  1.847250e+05  1.831640e+05    21
6     IG01         JVM-Threads_7778_JVM_ThreadCount_Threads  7.400000e+01  6.700000e+01    31
7     IG01                  OSLinux-CPU_CPU-0_SingleCpuUtil  1.813950e+00  1.231200e+00    11
8     IG01                  OSLinux-CPU_CPU-0_SingleCpuidle  9.875529e+01  9.807476e+01    15
9     IG01                  OSLinux-CPU_CPU-2_SingleCpuUtil  1.764500e+00  8.151000e-01    11
10    IG01                  OSLinux-CPU_CPU-2_SingleCpuidle  9.914487e+01  9.826911e+01    15
11    IG01                       OSLinux-CPU_CPU_CPUCpuUtil  2.618510e+01  2.580580e+01    21
12    IG01                          OSLinux-CPU_CPU_CPULoad  1.135000e+00  1.000000e+00    31
13    IG01                       OSLinux-CPU_CPU_CPUSysTime  4.907030e-01  3.181010e-01    31
14    IG01                      OSLinux-CPU_CPU_CPUUserTime  2.535451e+01  2.526045e+01    31
15    IG01                           OSLinux-CPU_CPU_CPUWio  2.100000e-01  1.453850e-02    31
16    IG01                      OSLinux-CPU_CPU_CPUidleutil  7.417760e+01  7.382419e+01    31
17    IG01    OSLinux-OSLinux_FILESYSTEM_-_FSAvailableSpace  9.953000e+03  9.953000e+03    11
18    IG01          OSLinux-OSLinux_FILESYSTEM_-_FSCapacity  3.025433e+01  3.025420e+01    15
19    IG01  OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent  1.636640e+01  1.636637e+01    15, (28015, 5))```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse in-memory 'trace' if available, otherwise load from CSV
try:
    trace
except NameError:
    trace = pd.read_csv('trace.csv')

# Ensure numeric value column
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')

# 1) Compute global P95 and P05 per (cmdb_id, trace_name)
trace_thresholds = (
    trace
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(p95=lambda s: s.quantile(0.95), p05=lambda s: s.quantile(0.05), rows='size')
    .reset_index()
)

# 2) Filter trace rows to incident window and store
trace_incident_filtered = trace[
    (trace['timestamp'] >= 1615104000) & (trace['timestamp'] <= 1615105800)
].copy()
trace_incident_filtered['dt'] = pd.to_datetime(trace_incident_filtered['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 3) Identify anomaly rows by joining thresholds
t = trace_incident_filtered.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')

# Determine anomaly_type
t['anomaly_type'] = np.where(t['value'] >= t['p95'], 'HIGH',
                             np.where(t['value'] <= t['p05'], 'LOW', None))

# Compute severity_ratio vectorized, handling zero/NaN denominators
# For HIGH: severity = value / p95  (if p95 > 0)
# For LOW:  severity = p05 / value  (if value > 0)
p95 = t['p95'].values
p05 = t['p05'].values
val = t['value'].values
atype = t['anomaly_type'].values

# initialize severity with NaN
severity = np.full(len(t), np.nan, dtype=float)

# HIGH
high_mask = (atype == 'HIGH') & (p95 != 0) & (~np.isnan(p95)) & (~np.isnan(val))
severity[high_mask] = val[high_mask] / p95[high_mask]

# LOW
low_mask = (atype == 'LOW') & (val != 0) & (~np.isnan(p05)) & (~np.isnan(val))
severity[low_mask] = p05[low_mask] / val[low_mask]

t['severity_ratio'] = severity

anomalies = t[t['anomaly_type'].notna()].copy()

# 4) Aggregate anomalies by (cmdb_id, trace_name, anomaly_type)
# rows_in_window = total rows for that (cmdb_id,trace_name) within filtered window
rows_in_window_df = trace_incident_filtered.groupby(['cmdb_id','trace_name']).size().reset_index().rename(columns={0:'rows_in_window'})

trace_anom_agg = (
    anomalies
    .groupby(['cmdb_id','trace_name','anomaly_type'])
    .agg(
        anomaly_count=('severity_ratio','size'),
        earliest_anomaly_timestamp=('timestamp','min'),
        max_severity_ratio=('severity_ratio','max')
    )
    .reset_index()
)

trace_anom_agg = trace_anom_agg.merge(rows_in_window_df, on=['cmdb_id','trace_name'], how='left')

# 5) Top 20 by max_severity_ratio desc, keep requested columns
trace_anomalies_agg_top20 = trace_anom_agg.sort_values('max_severity_ratio', ascending=False).reset_index(drop=True)
trace_anomalies_agg_top20 = trace_anomalies_agg_top20[
    ['cmdb_id','trace_name','anomaly_type','anomaly_count','earliest_anomaly_timestamp','max_severity_ratio','rows_in_window']
].head(20)

# Display only the compact aggregated anomalies table (top 20)
trace_anomalies_agg_top20

```
Out[3]:
```
```python
# Prepare a plain-English summary string based on the in-memory results and display it.
try:
    n_thresholds = trace_thresholds.shape[0]
except NameError:
    n_thresholds = 'unknown'

try:
    rows_in_window = trace_incident_filtered.shape[0]
except NameError:
    rows_in_window = 'unknown'

try:
    agg_df = trace_anomalies_agg_top20
    n_groups = agg_df.shape[0]
except NameError:
    agg_df = None
    n_groups = 0

top_lines = []
if agg_df is not None and n_groups > 0:
    for i, row in agg_df.head(5).iterrows():
        top_lines.append(
            f"{i+1}. {row['cmdb_id']} - {row['trace_name']} [{row['anomaly_type']}], "
            f"count={int(row['anomaly_count'])}, earliest_ts={int(row['earliest_anomaly_timestamp'])}, "
            f"max_severity={float(row['max_severity_ratio']):.3f}, rows_in_window={int(row['rows_in_window'])}"
        )
else:
    top_lines.append("No anomalies detected in the incident window.")

summary = (
    f"Trace anomaly detection summary:\n"
    f"- Thresholds computed for {n_thresholds} unique (cmdb_id, trace_name) pairs and saved in 'trace_thresholds'.\n"
    f"- Trace rows in the incident window (timestamps 1615104000..1615105800): {rows_in_window} rows saved in 'trace_incident_filtered'.\n"
    f"- Aggregated anomaly groups identified in the window (returned up to top 20): {n_groups} groups.\n"
    f"- Top anomaly groups by severity (top 5):\n  " + "\n  ".join(top_lines) + "\n\n"
    f"Notes:\n"
    f"- HIGH anomalies: value >= global P95; LOW anomalies: value <= global P05 (thresholds computed from full historical series).\n"
    f"- The aggregated anomaly table (up to 20 rows) is available as 'trace_anomalies_agg_top20' in the kernel for follow-up, "
    f"and thresholds/filtered data are in 'trace_thresholds' and 'trace_incident_filtered'."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                         trace_name anomaly_type  anomaly_count  earliest_anomaly_timestamp  max_severity_ratio  rows_in_window
0       IG02               trace.self.row_count          LOW              2                  1615105740          651.500000              31
1   dockerA1            trace.self.duration_p95         HIGH              2                  1615105440           57.059490              30
2   dockerB1       trace.from_root.duration_p95          LOW              1                  1615105500           31.620000               2
3       MG01            trace.self.duration_p95          LOW              2                  1615104480           28.095541              30
4   Tomcat04         trace.to_MG01.duration_p95          LOW              2                  1615104480           27.405333              29
5       MG01   trace.from_Tomcat04.duration_p95          LOW              2                  1615104480           27.405333              29
6   Tomcat02         trace.to_MG01.duration_p95          LOW              2                  1615104480           20.531061              29
7       MG01   trace.from_Tomcat02.duration_p95          LOW              2                  1615104480           20.531061              29
8       IG02            trace.self.duration_p95          LOW              2                  1615105260           20.433621              31
9       MG01   trace.from_Tomcat03.duration_p95          LOW              2                  1615104540           19.524349              30
10  Tomcat03         trace.to_MG01.duration_p95          LOW              2                  1615104540           19.524349              30
11      IG01            trace.self.duration_p95         HIGH              2                  1615104240           19.217803              30
12      IG02            trace.self.duration_p95         HIGH              2                  1615104240           19.188487              31
13      MG01   trace.from_Tomcat01.duration_p95          LOW              2                  1615104660           18.042680              30
14  Tomcat01         trace.to_MG01.duration_p95          LOW              2                  1615104660           18.042680              30
15      MG01     trace.to_dockerA2.duration_p95          LOW              2                  1615104480           14.765253              30
16  dockerA2       trace.from_MG01.duration_p95          LOW              2                  1615104480           14.765253              30
17      IG02           trace.self.duration_mean         HIGH              2                  1615104720            9.964907              31
18      IG01           trace.self.duration_mean         HIGH              2                  1615104720            9.640061              30
19      MG01  trace.from_Tomcat02.duration_mean          LOW              2                  1615104480            9.439124              29```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Candidate components
candidates = ['Tomcat04', 'apache01', 'apache02', 'IG02', 'MG01', 'MG02', 'Mysql02', 'Tomcat01']

# Ensure 'log' and 'error_logs' exist in kernel; if not, load them
try:
    log
except NameError:
    log = pd.read_csv('log.csv')
try:
    error_logs
except NameError:
    error_logs = pd.read_csv('error_logs.csv')

# Ensure numeric 'value' in logs
if 'value' in log.columns:
    log['value'] = pd.to_numeric(log['value'], errors='coerce')

# Time window filter
start_ts = 1615104000
end_ts = 1615105800

# 1) Filter logs for window and candidate components, save to kernel variable
log_incident_filtered = log[
    (log['timestamp'] >= start_ts) & (log['timestamp'] <= end_ts) & (log['cmdb_id'].isin(candidates))
].copy()

# add localized datetime column for convenience
if 'timestamp' in log_incident_filtered.columns:
    log_incident_filtered['dt'] = pd.to_datetime(log_incident_filtered['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 2) Aggregate filtered logs by (cmdb_id, log_name)
if not log_incident_filtered.empty:
    logs_agg = (
        log_incident_filtered
        .groupby(['cmdb_id', 'log_name'])
        .agg(
            rows_in_window=('value', 'size'),
            earliest_timestamp=('timestamp', 'min'),
            sum_value=('value', 'sum'),
            max_value=('value', 'max')
        )
        .reset_index()
    )
    # keep compact and deterministic ordering: sort by rows_in_window desc then max_value desc
    logs_agg_top20 = logs_agg.sort_values(['rows_in_window','max_value'], ascending=[False, False]).head(20).reset_index(drop=True)
else:
    # empty result placeholder
    logs_agg_top20 = pd.DataFrame(columns=['cmdb_id','log_name','rows_in_window','earliest_timestamp','sum_value','max_value'])

# 3) Filter error_logs for window and candidate components, save to kernel variable
error_logs_incident = error_logs[
    (error_logs['timestamp'] >= start_ts) & (error_logs['timestamp'] <= end_ts) & (error_logs['cmdb_id'].isin(candidates))
].copy()

# add localized datetime column if not present
if 'timestamp' in error_logs_incident.columns:
    error_logs_incident['dt'] = pd.to_datetime(error_logs_incident['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# compact view limited to 50 rows
error_logs_incident_head = error_logs_incident[['timestamp','cmdb_id','message','dt']].head(50)

# Return the compact aggregated logs table (max 20 rows) and the filtered error logs (up to 50 rows).
logs_agg_top20, error_logs_incident_head

```
Out[4]:
```
```python
summary = (
    "Log & error-log incident window summary (timestamps 1615104000..1615105800) for selected components:\n\n"
    "Logs (aggregated by cmdb_id, log_name):\n"
    "- High volume of log row counts during the window:\n"
    "  * apache02 - log.row_count: 31 samples, sum=27298, max=1082 (earliest ts=1615104000)\n"
    "  * apache01 - log.row_count: 31 samples, sum=21387, max=747 (earliest ts=1615104000)\n"
    "  * Tomcat04 - log.row_count: 31 samples, sum=12215, max=497 (earliest ts=1615104000)\n"
    "  * Tomcat01 - log.row_count: 31 samples, sum=12211, max=449 (earliest ts=1615104000)\n"
    "- Error counts in window:\n"
    "  * apache02 - log.error_count: 31 samples, sum=26 (max=26)\n"
    "  * apache01 - log.error_count: 31 samples, sum=25 (max=25)\n"
    "  * Tomcat04 - log.error_count: 31 samples, sum=1  (max=1)\n"
    "  * Tomcat01 - log.error_count: 31 samples, sum=0  (max=0)\n"
    "- GC pause metrics present for Tomcat nodes (partial): Tomcat01 and Tomcat04 have log.gc_pause_max rows in-window.\n\n"
    "Error logs (filtered rows, up to 50 shown):\n"
    "- Error-log entries from Tomcat04, apache01, apache02 are present in the incident window (examples around 2021-03-07 16:13–16:29+08:00).\n"
    "- 50 rows were returned for inspection (variable 'error_logs_incident' saved in kernel).\n\n"
    "Saved kernel variables for follow-up:\n"
    "- 'log_incident_filtered' contains the filtered log rows for the candidate components and time window.\n"
    "- 'error_logs_incident' contains the filtered error log rows for the same components and window.\n\n"
    "Notes:\n"
    "- The aggregated log table returned is compact (max 20 rows) and shows counts, earliest timestamps, sums and maxima for each (cmdb_id, log_name).\n"
    "- If you want deeper inspection (e.g., view more rows or inspect specific timestamps/messages), request df.head(N) or specific component details."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id          log_name  rows_in_window  earliest_timestamp  sum_value  max_value
0  apache02     log.row_count              31          1615104000   27298.00    1082.00
1  apache01     log.row_count              31          1615104000   21387.00     747.00
2  Tomcat04     log.row_count              31          1615104000   12215.00     497.00
3  Tomcat01     log.row_count              31          1615104000   12211.00     449.00
4  apache02   log.error_count              31          1615104000      26.00      26.00
5  apache01   log.error_count              31          1615104000      25.00      25.00
6  Tomcat04   log.error_count              31          1615104000       1.00       1.00
7  Tomcat01   log.error_count              31          1615104000       0.00       0.00
8  Tomcat01  log.gc_pause_max              21          1615104000       1.51       0.10
9  Tomcat04  log.gc_pause_max              21          1615104060       1.33       0.08,      timestamp   cmdb_id                                            message                        dt
0   1615104822  Tomcat04  IG02 POST /UOCP/person/ServiceTest9.json HTTP/... 2021-03-07 16:13:42+08:00
1   1615105743  apache01  IPAddress "POST /UOCP/person/ServiceTest10.jso... 2021-03-07 16:29:03+08:00
2   1615105743  apache02  IPAddress "POST /UOCP/person/ServiceTest1.json... 2021-03-07 16:29:03+08:00
3   1615105745  apache02  IPAddress "POST /UOCP/person/ServiceTest5.json... 2021-03-07 16:29:05+08:00
4   1615105746  apache01  IPAddress "POST /UOCP/base/ServiceTest6.json H... 2021-03-07 16:29:06+08:00
..         ...       ...                                                ...                       ...
45  1615105789  apache01  IPAddress "POST /UOCP/person/ServiceTest9.json... 2021-03-07 16:29:49+08:00
46  1615105790  apache02  IPAddress "POST /UOCP/person/ServiceTest3.json... 2021-03-07 16:29:50+08:00
47  1615105794  apache01  IPAddress "POST /UOCP/person/ServiceTest3.json... 2021-03-07 16:29:54+08:00
48  1615105795  apache02  IPAddress "POST /UOCP/person/ServiceTest9.json... 2021-03-07 16:29:55+08:00
49  1615105796  apache01  IPAddress "POST /UOCP/person/ServiceTest9.json... 2021-03-07 16:29:56+08:00

[50 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import numpy as np
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variables:
# metric_thresholds, metric_incident_filtered, trace_thresholds, trace_incident_filtered

# Candidate components
candidates = ['MG01','MG02','Tomcat04','Tomcat01','apache01','apache02','IG01','IG02','Mysql02','Tomcat03','Tomcat02','Redis01']

# Incident windows
A_start, A_end = 1615104660, 1615104860
B_start, B_end = 1615105680, 1615105800

def build_evidence(window_start, window_end):
    # Metrics: filter incident window (metric_incident_filtered already within broader incident)
    m_window = metric_incident_filtered[
        (metric_incident_filtered['timestamp'] >= window_start) &
        (metric_incident_filtered['timestamp'] <= window_end) &
        (metric_incident_filtered['cmdb_id'].isin(candidates))
    ].copy()
    # Join thresholds
    m = m_window.merge(metric_thresholds, left_on=['cmdb_id','kpi_name'], right_on=['cmdb_id','kpi_name'], how='left')
    # Determine anomaly types
    m['anomaly_type'] = np.where(m['value'] >= m['p95'], 'HIGH',
                                 np.where(m['value'] <= m['p05'], 'LOW', None))
    m = m[m['anomaly_type'].notna()].copy()
    if not m.empty:
        # severity_ratio safely
        m['severity_ratio'] = np.nan
        high_mask = (m['anomaly_type']=='HIGH') & (m['p95'].notna()) & (m['p95'] != 0)
        low_mask  = (m['anomaly_type']=='LOW')  & (m['value'].notna()) & (m['value'] != 0)
        m.loc[high_mask, 'severity_ratio'] = m.loc[high_mask, 'value'] / m.loc[high_mask, 'p95']
        m.loc[low_mask,  'severity_ratio'] = m.loc[low_mask,  'p05'] / m.loc[low_mask,  'value']
        # Keep earliest anomalous row per (cmdb_id,kpi_name,anomaly_type)
        m_sorted = m.sort_values('timestamp')
        m_earliest = m_sorted.drop_duplicates(subset=['cmdb_id','kpi_name','anomaly_type'], keep='first')
        metric_evidence = m_earliest[[
            'cmdb_id'
        ]].copy()
        metric_evidence['source'] = 'metric'
        metric_evidence['name'] = m_earliest['kpi_name']
        metric_evidence['anomaly_type'] = m_earliest['anomaly_type']
        metric_evidence['earliest_timestamp'] = m_earliest['timestamp']
        metric_evidence['value'] = m_earliest['value']
        metric_evidence['threshold_p95'] = m_earliest['p95']
        metric_evidence['threshold_p05'] = m_earliest['p05']
        metric_evidence['severity_ratio'] = m_earliest['severity_ratio']
    else:
        metric_evidence = pd.DataFrame(columns=['cmdb_id','source','name','anomaly_type','earliest_timestamp','value','threshold_p95','threshold_p05','severity_ratio'])

    # Traces: similar process
    t_window = trace_incident_filtered[
        (trace_incident_filtered['timestamp'] >= window_start) &
        (trace_incident_filtered['timestamp'] <= window_end) &
        (trace_incident_filtered['cmdb_id'].isin(candidates))
    ].copy()
    t = t_window.merge(trace_thresholds, left_on=['cmdb_id','trace_name'], right_on=['cmdb_id','trace_name'], how='left')
    t['anomaly_type'] = np.where(t['value'] >= t['p95'], 'HIGH',
                                 np.where(t['value'] <= t['p05'], 'LOW', None))
    t = t[t['anomaly_type'].notna()].copy()
    if not t.empty:
        t['severity_ratio'] = np.nan
        high_mask = (t['anomaly_type']=='HIGH') & (t['p95'].notna()) & (t['p95'] != 0)
        low_mask  = (t['anomaly_type']=='LOW')  & (t['value'].notna()) & (t['value'] != 0)
        t.loc[high_mask, 'severity_ratio'] = t.loc[high_mask, 'value'] / t.loc[high_mask, 'p95']
        t.loc[low_mask,  'severity_ratio'] = t.loc[low_mask,  'p05'] / t.loc[low_mask,  'value']
        t_sorted = t.sort_values('timestamp')
        t_earliest = t_sorted.drop_duplicates(subset=['cmdb_id','trace_name','anomaly_type'], keep='first')
        trace_evidence = t_earliest[[
            'cmdb_id'
        ]].copy()
        trace_evidence['source'] = 'trace'
        trace_evidence['name'] = t_earliest['trace_name']
        trace_evidence['anomaly_type'] = t_earliest['anomaly_type']
        trace_evidence['earliest_timestamp'] = t_earliest['timestamp']
        trace_evidence['value'] = t_earliest['value']
        trace_evidence['threshold_p95'] = t_earliest['p95']
        trace_evidence['threshold_p05'] = t_earliest['p05']
        trace_evidence['severity_ratio'] = t_earliest['severity_ratio']
    else:
        trace_evidence = pd.DataFrame(columns=['cmdb_id','source','name','anomaly_type','earliest_timestamp','value','threshold_p95','threshold_p05','severity_ratio'])

    # Combine metric and trace evidence
    combined = pd.concat([metric_evidence, trace_evidence], ignore_index=True, sort=False)
    # Keep specified column order and sort by severity descending, limit to top 20
    if not combined.empty:
        combined = combined[[
            'cmdb_id','source','name','anomaly_type','earliest_timestamp','value','threshold_p95','threshold_p05','severity_ratio'
        ]]
        combined_sorted = combined.sort_values('severity_ratio', ascending=False).head(20).reset_index(drop=True)
    else:
        combined_sorted = combined

    return combined_sorted

# Build evidence for Incident A and B and save to kernel variables
incident_A_evidence = build_evidence(A_start, A_end)
incident_B_evidence = build_evidence(B_start, B_end)

# Return the compact tables (top 20 each)
incident_A_evidence, incident_B_evidence

```
Out[5]:
```
```python
summary = (
    "Incident evidence summary (plain English):\n\n"
    "Incident A (16:11–16:14, ts 1615104660–1615104860):\n"
    "- Strongest signals are from MG01 across traces and metrics. Top items:\n"
    "  * MG01 trace.self.duration_p95 (LOW) — earliest 1615104660, very large severity (~28x).\n"
    "  * MG01 OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec (LOW) — earliest 1615104720, severity ~26x.\n"
    "  * MG02 OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT (HIGH) — earliest 1615104780, severity 20x.\n"
    "  * Multiple MG01 trace links (to/from dockerA*, Tomcat*) show LOW duration_p95 or row_count anomalies (severity 3–15x).\n"
    "- Overall impression: MG01 exhibits concentrated latency/network-related degradations in this window, with supporting metric drops in network throughput/packets.\n\n"
    "Incident B (16:28–16:30, ts 1615105680–1615105800):\n"
    "- Top metric anomalies are heavy TCP-FIN-WAIT counts on IG02 and Tomcat04:\n"
    "  * IG02 OSLinux-..._TCP-FIN-WAIT (HIGH) — earliest 1615105800, severity 124x (highest overall).\n"
    "  * Tomcat04 OSLinux-..._TCP-FIN-WAIT (HIGH) — earliest 1615105800, severity 51x.\n"
    "- Trace anomalies indicate IG01/IG02 experienced large trace.duration_p95 and duration_mean spikes (~19x and ~9x severity), plus reduced row_counts on several paths involving Tomcat04/MG02.\n"
    "  * IG01 trace.self.duration_p95 (HIGH) — earliest 1615105740, severity ~19x.\n"
    "  * IG02 trace.self.duration_p95 and duration_mean (HIGH) — earliest 1615105740/1615105740, severe.\n"
    "- Additional supporting signals: some local disk and network metrics on apache01/IG01/MG02 show abnormal values (moderate severity).\n\n"
    "Stored kernel variables for follow-up:\n"
    "- 'incident_A_evidence' contains the compact top-evidence table for Incident A.\n"
    "- 'incident_B_evidence' contains the compact top-evidence table for Incident B.\n\n"
    "Conclusion (straightforward):\n"
    "- Incident A: MG01 is the dominant culprit (trace latency and network throughput/packet anomalies).\n"
    "- Incident B: IG02 and Tomcat04 show the strongest metric anomalies (very high TCP-FIN-WAIT), with IG01/IG02 trace latency spikes—suggesting upstream service/connection issues around those components.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id  source                                             name anomaly_type  earliest_timestamp       value  threshold_p95  threshold_p05  severity_ratio
0       MG01   trace                          trace.self.duration_p95          LOW          1615104660    0.037000       2.098315       1.039535       28.095541
1       MG01  metric  OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec          LOW          1615104720  348.000000  812468.400000    9124.800000       26.220690
2       MG02  metric   OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT         HIGH          1615104780    1.000000       0.050000       0.000000       20.000000
3       MG01   trace                   trace.to_dockerA2.duration_p95          LOW          1615104660    0.033600       2.092050       0.496113       14.765253
4       MG01  metric      OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn          LOW          1615104720   26.000000    1204.600000     340.000000       13.076923
5       MG01  metric     OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut          LOW          1615104720   25.000000    1130.800000     317.400000       12.696000
6   Tomcat03   trace                       trace.to_MG01.duration_p95          LOW          1615104660    0.039000       3.131277       0.487133       12.490577
7       MG01   trace                 trace.from_Tomcat03.duration_p95          LOW          1615104660    0.039000       3.131277       0.487133       12.490577
8   Tomcat01   trace                       trace.to_MG01.duration_p95          LOW          1615104660    0.030450       2.906290       0.326573       10.724877
9       MG01   trace                 trace.from_Tomcat01.duration_p95          LOW          1615104660    0.030450       2.906290       0.326573       10.724877
10      MG01  metric     OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT         HIGH          1615104720   19.000000       1.900000       0.000000       10.000000
11      MG01   trace                         trace.self.duration_mean          LOW          1615104660    0.017833       0.409295       0.133947        7.511165
12      MG01   trace                  trace.to_dockerA2.duration_mean          LOW          1615104660    0.030000       0.897575       0.163653        5.455113
13      MG01   trace                             trace.self.row_count          LOW          1615104660    6.000000    1358.100000      25.200000        4.200000
14      MG01   trace                trace.from_dockerA1.duration_mean         HIGH          1615104720    0.512500       0.128337       0.006242        3.993377
15      MG01   trace                    trace.from_dockerB1.row_count          LOW          1615104660    1.000000     248.050000       3.800000        3.800000
16  Tomcat01  metric     OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT         HIGH          1615104780    6.000000       1.750000       0.000000        3.428571
17      MG01   trace                      trace.to_dockerA1.row_count          LOW          1615104660    1.000000     201.950000       3.350000        3.350000
18      MG01   trace                    trace.from_Tomcat03.row_count          LOW          1615104660    1.000000     227.550000       3.350000        3.350000
19  Tomcat03   trace                          trace.to_MG01.row_count          LOW          1615104660    1.000000     227.550000       3.350000        3.350000,      cmdb_id  source                                               name anomaly_type  earliest_timestamp       value  threshold_p95  threshold_p05  severity_ratio
0       IG02  metric       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT         HIGH          1615105800   62.000000       0.500000       0.000000      124.000000
1   Tomcat04  metric       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT         HIGH          1615105800   51.000000       1.000000       0.000000       51.000000
2       IG02   trace                            trace.self.duration_p95          LOW          1615105800    0.058000       3.126875       1.185150       20.433621
3       IG01   trace                            trace.self.duration_p95         HIGH          1615105740   61.004850       3.174392       1.583478       19.217803
4       IG02   trace                            trace.self.duration_p95         HIGH          1615105740   60.000000       3.126875       1.185150       19.188487
5       IG02   trace                           trace.self.duration_mean         HIGH          1615105740    6.903000       0.692731       0.314773        9.964907
6       IG01   trace                           trace.self.duration_mean         HIGH          1615105740    6.826283       0.708116       0.356652        9.640061
7   apache01  metric     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT          LOW          1615105800    1.000000      29.600000       6.000000        6.000000
8       IG02   trace                           trace.self.duration_mean          LOW          1615105800    0.058000       0.692731       0.314773        5.427112
9       IG02   trace                        trace.to_Tomcat04.row_count          LOW          1615105740   45.000000     238.850000     181.900000        4.042222
10  Tomcat04   trace                          trace.from_IG02.row_count          LOW          1615105740   45.000000     238.850000     181.900000        4.042222
11  Tomcat04   trace                            trace.to_MG02.row_count          LOW          1615105740   49.000000     417.150000     174.250000        3.556122
12      MG02   trace                      trace.from_Tomcat04.row_count          LOW          1615105740   49.000000     417.150000     174.250000        3.556122
13  Tomcat04   trace                               trace.self.row_count          LOW          1615105740  712.000000    3172.950000    2440.950000        3.428301
14      IG01  metric  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...         HIGH          1615105740    0.200000       0.060000       0.000000        3.333333
15  Tomcat04   trace                          trace.from_IG01.row_count          LOW          1615105740   58.000000     240.200000     175.700000        3.029310
16      IG01   trace                        trace.to_Tomcat04.row_count          LOW          1615105740   58.000000     240.200000     175.700000        3.029310
17  apache01  metric     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps         HIGH          1615105740    2.000000       0.700000       0.000000        2.857143
18  Tomcat03  metric                             OSLinux-CPU_CPU_CPUWio         HIGH          1615105680    0.536600       0.255800       0.019750        2.097733
19  Tomcat04  metric  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...          LOW          1615105800  100.000000     196.000000     196.000000        1.960000)```
```